# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Andrey Pryakhin
**Student ID:** S23068249
**Group:** 10
**Repository:** https://github.com/pryakhin-andrew/hw3-pryakhin-andrew

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> I used Claude Code (Anthropic, model Claude Opus 5.5). It helped me write the code and the first draft of the written answers. All numbers, chunk ids and model answers in the text come from the outputs of this notebook, and I checked them against those outputs.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = **8.333**

**Query terms:** how, does, rag, retrieve, information

**TF, DF and IDF of each query term:** N = 3. A term that is in no document has n(t) = 0, so idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(7) = 1.9459.

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

> No term contributes to any document. Every f(t, d) is 0, so every term gives idf · 0 · 2.5 / (0 + …) = 0.
>
> score(Q, D1) = 0 + 0 + 0 + 0 + 0 = **0**
> score(Q, D2) = 0 + 0 + 0 + 0 + 0 = **0**
> score(Q, D3) = 0 + 0 + 0 + 0 + 0 = **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** 25 / 3 = **8.333** (stemming does not change the number of tokens)

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5/0.5) = 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | **2** | 0 | 1 | ln(2.5/1.5) = **0.5108** |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document**, worked:

> Only `retriev` in D2 gives a non-zero part.
>
> Length part for D2: k1 · (1 − b + b · |d| / avgdl) = 1.5 · (0.25 + 0.75 · 10 / 8.333) = 1.5 · (0.25 + 0.9) = 1.5 · 1.15 = 1.725
>
> retriev in D2: 0.5108 · 2 · 2.5 / (2 + 1.725) = 2.554 / 3.725 = **0.6857**
>
> score(Q, D1) = 0 (no query term appears; `gener` is in D1 but not in the query)
> score(Q, D2) = 0 + 0 + 0 + 0.6857 + 0 = **0.6857**
> score(Q, D3) = 0

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.6857 | 1 |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> In (a) BM25 retrieves nothing useful: all three scores are 0, so it is a three-way tie. The result looks like a mistake, but it is correct. BM25 only counts exact tokens, and not one query token is in any document. The query says `retrieve`, D2 says `retrieves` and `retrieval`. The query says `RAG`, D2 says `Retrieval-Augmented Generation`, which is the same thing written out in full. `how`, `does` and `information` are in no document. A library would just return the tie in index order, so D1 would come first, which is wrong.
>
> In (b) D2 is first with 0.6857. Stemming turns `retrieve`, `retrieves` and `retrieval` into the same token `retriev`, so the query finally shares one token with D2 (twice). `RAG` still matches nothing, because no stem rule links an abbreviation to its full form. The idf is small (0.51) because `retriev` is in one of three documents, which is a third of this tiny collection.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | √(0.49 + 0.01 + 0.16) = √0.66 = 0.8124 | 0.78 / (0.9644 · 0.8124) = 0.78 / 0.7835 = **0.9956** |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | √(0.81 + 0.09 + 0.36) = √1.26 = 1.1225 | 1.08 / (0.9644 · 1.1225) = 1.08 / 1.0825 = **0.9977** |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | √(0.01 + 0.64 + 0.04) = √0.69 = 0.8307 | 0.34 / (0.9644 · 0.8307) = 0.34 / 0.8011 = **0.4244** |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2, with 0.9977. D1 is very close behind (0.9956). D3 is far away (0.4244).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Not with (a). In (a) BM25 gives 0 to everything, so it has no ranking at all, while cosine puts D2 first.
> With (b) the top document is the same (D2), but the rest is different. BM25 (b) gives D1 and D3 the same score of 0. Cosine puts D1 almost level with D2 and puts D3 clearly last.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 compares letters, embeddings compare meaning. The query says `RAG` and `retrieve information`. D2 says `Retrieval-Augmented Generation retrieves relevant documents`. For a person this is the same idea: RAG is the short name, `retrieves` is a form of `retrieve`, and documents carry information. BM25 sees zero shared tokens (and only one after stemming). An embedding model has learned that these words are used in the same places, so it gives them close vectors. That is also why D1 (`answer questions`, `language models`) gets a high cosine but 0 from BM25: it is about the same topic with no shared words. D3 (`numerical vectors`) is a different topic, so both methods put it low.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
import math
import re

import numpy as np

A_DOCS = {
    "D1": "Large language models can generate text and answer questions.",
    "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
    "D3": "Embeddings represent text as numerical vectors.",
}
A_QUERY = "How does RAG retrieve information?"
A_STEMS = {
    "retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
    "generate": "gener", "generating": "gener", "generation": "gener",
    "information": "inform", "does": "doe",
}


def a_tokenize(text, stem=False):
    tokens = [t for t in re.split(r"[ \-]", re.sub(r"[^\w\s-]", "", text.lower())) if t]
    return [A_STEMS.get(t, t) for t in tokens] if stem else tokens


def bm25_by_hand(docs, query, k1=1.5, b=0.75):
    n_docs = len(docs)
    avgdl = sum(map(len, docs.values())) / n_docs
    scores = {}
    for name, tokens in docs.items():
        score = 0.0
        for term in query:
            n = sum(term in d for d in docs.values())
            idf = math.log((n_docs - n + 0.5) / (n + 0.5))
            f = tokens.count(term)
            score += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(tokens) / avgdl))
        scores[name] = score
    return avgdl, scores


for stem in (False, True):
    docs = {name: a_tokenize(text, stem) for name, text in A_DOCS.items()}
    query = a_tokenize(A_QUERY, stem)
    avgdl, scores = bm25_by_hand(docs, query)
    print(f"A1 ({'b' if stem else 'a'}) query={query} avgdl={avgdl:.4f}")
    for name, tokens in docs.items():
        print(f"  {name} |d|={len(tokens):2d} BM25={scores[name]:.4f}")

a_query_vec = np.array([0.8, 0.2, 0.5])
a_doc_vecs = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
print("A2")
for name, vec in a_doc_vecs.items():
    vec = np.array(vec)
    dot, nq, nd = a_query_vec @ vec, np.linalg.norm(a_query_vec), np.linalg.norm(vec)
    print(f"  {name} dot={dot:.2f} |Q|={nq:.4f} |D|={nd:.4f} cos={dot / (nq * nd):.4f}")

A1 (a) query=['how', 'does', 'rag', 'retrieve', 'information'] avgdl=8.3333
  D1 |d|= 9 BM25=0.0000
  D2 |d|=10 BM25=0.0000
  D3 |d|= 6 BM25=0.0000
A1 (b) query=['how', 'doe', 'rag', 'retriev', 'inform'] avgdl=8.3333
  D1 |d|= 9 BM25=0.0000
  D2 |d|=10 BM25=0.6857
  D3 |d|= 6 BM25=0.0000
A2
  D1 dot=0.78 |Q|=0.9644 |D|=0.8124 cos=0.9956
  D2 dot=1.08 |Q|=0.9644 |D|=1.1225 cos=0.9977
  D3 dot=0.34 |Q|=0.9644 |D|=0.8307 cos=0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import json
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path(".env"))

DATA = Path("data")
corpus = [json.loads(line) for line in (DATA / "corpus.jsonl").read_text(encoding="utf-8").splitlines()]
queries = json.loads((DATA / "queries.json").read_text(encoding="utf-8"))["queries"]
qa = json.loads((DATA / "qa.json").read_text(encoding="utf-8"))["questions"]
qa_by_id = {q["id"]: q for q in qa}

print(f"{len(corpus)} chunks, {len(queries)} fixed queries, {len(qa)} QA questions")
print({lang: sum(c["lang"] == lang for c in corpus) for lang in ("en", "ru", "kk")})

74 chunks, 8 fixed queries, 6 QA questions
{'en': 71, 'ru': 2, 'kk': 1}


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

> Lowercase, then `re.findall(r"\w+", text)`. In Python 3 `\w` is Unicode, so Russian and Kazakh letters are kept as words. No stemming and no stop word removal. Effects I need later: `GR-305` becomes two tokens `gr` and `305`. `master's` becomes `master` and `s`. Common words like `the`, `is`, `to`, `can` still count a little, and in a Russian query `и` and `в` count too. BM25Okapi with `k1=1.5, b=0.75`.

In [3]:
from rank_bm25 import BM25Okapi

K1, B = 1.5, 0.75


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(c["text"]) for c in corpus], k1=K1, b=B)


def top_k(scores, k):
    return [(corpus[i], float(scores[i])) for i in np.argsort(-scores, kind="stable")[:k]]


def bm25_search(query, k=3):
    return top_k(bm25.get_scores(tokenize(query)), k)


def bm25_terms(query, chunk_id):
    i = next(i for i, c in enumerate(corpus) if c["id"] == chunk_id)
    freqs, dl = bm25.doc_freqs[i], bm25.doc_len[i]
    parts = {}
    for term in tokenize(query):
        f = freqs.get(term, 0)
        if f:
            parts[term] = bm25.idf[term] * f * (K1 + 1) / (f + K1 * (1 - B + B * dl / bm25.avgdl))
    return {t: round(s, 2) for t, s in parts.items()}


def show(results):
    for rank, (chunk, score) in enumerate(results, 1):
        print(f"  {rank}. {chunk['id']:<6} {score:7.3f}  {chunk['title']} | {chunk['text'][:70]}…")


print(f"vocabulary: {len(bm25.idf)} terms, avgdl = {bm25.avgdl:.1f} tokens")
show(bm25_search(queries[0]["text"]))

vocabulary: 2160 terms, avgdl = 93.8 tokens
  1. GO-08   15.825  Form GR-305: appeals | Form GR-305 is used to appeal a decision of the Grant Office. It is th…
  2. GO-12    8.195  How to appeal a decision | A student who disagrees with a decision may appeal within 14 calendar …
  3. GO-06    8.070  Form GR-104: the application form | Form GR-104 is the application form for the need-based study grant. It…


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [4]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("google/embeddinggemma-300m")
doc_embeddings = embedder.encode_document([c["text"] for c in corpus])


def emb_search(query, k=3):
    sims = embedder.similarity(embedder.encode_query(query), doc_embeddings)[0].numpy()
    return top_k(sims, k)


print(f"doc embeddings: {doc_embeddings.shape}, dtype {doc_embeddings.dtype}")
show(emb_search(queries[0]["text"]))

/Users/kqart/Andrey/Code/Uni/Artificial intelligence/hw3-pryakhin-andrew/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 6558.72it/s]

doc embeddings: (74, 768), dtype float32
  1. GO-08    0.696  Form GR-305: appeals | Form GR-305 is used to appeal a decision of the Grant Office. It is th…
  2. GO-12    0.461  How to appeal a decision | A student who disagrees with a decision may appeal within 14 calendar …
  3. GO-23    0.450  Апелляция: кратко | Если студент не согласен с решением, он может подать апелляцию по форм…


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Who has to sign form GR-210? | GO-07 | **BM25.** `210` is a rare exact token, it is only in a few chunks. The three form names (GR-104, GR-210, GR-305) look almost the same, so I expected the embedding model to mix them up. |
| Q-10 | Где находится грантовый отдел и в какие часы он работает? (Where is the grant office and what are its opening hours?) | GO-21 | **Embeddings.** GO-21 is in English and the query is in Russian, so they share no word. BM25 can only match Russian words, and the only Russian chunks (GO-15, GO-23) are about payment and appeals. |
| Q-11 | When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish? | KZ-43 (from `qa.json`, C-04) | **BM25 on rank 1.** The rare words `Cyrillic`, `Latin` and `alphabet` are all in KZ-43. |

Note on Q-11: I added it after the first run, because Q-09 turned out to be a tie (both retrievers put GO-07 first). Its `relevant` comes from `qa.json`, not from me. I had already seen this question's retrieval in Part C, so this expectation was not blind.

In [5]:
own_queries = [
    {"id": "Q-09", "text": "Who has to sign form GR-210?", "relevant": ["GO-07"]},
    {"id": "Q-10", "text": "Где находится грантовый отдел и в какие часы он работает?", "relevant": ["GO-21"]},
    {"id": "Q-11", "text": qa_by_id["C-04"]["question"], "relevant": qa_by_id["C-04"]["relevant"]},
]
all_queries = queries + own_queries
for q in own_queries:
    print(q["id"], q["text"], q["relevant"])

Q-09 Who has to sign form GR-210? ['GO-07']
Q-10 Где находится грантовый отдел и в какие часы он работает? ['GO-21']
Q-11 When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish? ['KZ-43']


## Results for all ten or more queries: top 3 from each retriever

In [6]:
RETRIEVERS = {"BM25": bm25_search, "Embedding": emb_search}


def hit(results, relevant):
    return any(chunk["id"] in relevant for chunk, _ in results)


b_results = {}
for q in all_queries:
    print(f"{q['id']}: {q['text']}   relevant={q['relevant']}")
    for name, search in RETRIEVERS.items():
        results = search(q["text"])
        b_results[q["id"], name] = results
        print(f" {name}  hit@3 = {'✓' if hit(results, q['relevant']) else '✗'}")
        show(results)
    print()

Q-01: What is form GR-305 used for?   relevant=['GO-08', 'GO-12', 'GO-23']
 BM25  hit@3 = ✓
  1. GO-08   15.825  Form GR-305: appeals | Form GR-305 is used to appeal a decision of the Grant Office. It is th…
  2. GO-12    8.195  How to appeal a decision | A student who disagrees with a decision may appeal within 14 calendar …
  3. GO-06    8.070  Form GR-104: the application form | Form GR-104 is the application form for the need-based study grant. It…
 Embedding  hit@3 = ✓
  1. GO-08    0.696  Form GR-305: appeals | Form GR-305 is used to appeal a decision of the Grant Office. It is th…
  2. GO-12    0.461  How to appeal a decision | A student who disagrees with a decision may appeal within 14 calendar …
  3. GO-23    0.450  Апелляция: кратко | Если студент не согласен с решением, он может подать апелляцию по форм…

Q-02: My family is poor. Is there financial help for students like me?   relevant=['GO-01', 'GO-02', 'GO-03']
 BM25  hit@3 = ✗
  1. KZ-17    8.726  Lake Balkhash (part 2) 

 Embedding  hit@3 = ✓
  1. GO-01    0.445  About the need-based study grant 2026 | The need-based study grant 2026 is run by the Grant Office for full-ti…
  2. GO-10    0.412  What counts as proof of income | Proof of income must accompany form GR-210. Accepted: a salary certifi…
  3. GO-20    0.409  Who cannot apply | The need-based grant is for full-time undergraduate students only. Par…

Q-03: Өтінішті қашанға дейін тапсыру керек?   relevant=['GO-09', 'GO-14']
 BM25  hit@3 = ✓
  1. GO-14   10.698  Өтініш беру мерзімі және тәртібі | 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін сту…
  2. KZ-01    0.000  Kazakhstan (part 1) | Kazakhstan, officially the Republic of Kazakhstan, is a landlocked cou…
  3. KZ-02    0.000  Kazakhstan (part 2) | Kazakhstan is the world's ninth-largest country by land area and the l…


 Embedding  hit@3 = ✓
  1. GO-14    0.399  Өтініш беру мерзімі және тәртібі | 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін сту…
  2. GO-09    0.350  Application deadline | Applications for the 2026 grant open on 1 September 2026 and close on …
  3. GO-11    0.301  When a document is missing | If a required document is missing, the Grant Office does not refuse th…

Q-04: What is the minimum GPA for the grant?   relevant=['GO-02']
 BM25  hit@3 = ✗
  1. GO-05   12.780  Archived: the 2025 scheme (no longer in force) | ARCHIVED — kept for reference only. Under the 2025 scheme the minimum …
  2. GO-17   10.518  Academic progress during the year | The grant continues only while the student keeps up their studies. If …
  3. GO-19   10.136  Priority when the budget runs short | The grant budget is fixed each year. If there are more qualifying appl…
 Embedding  hit@3 = ✓
  1. GO-02    0.665  Who qualifies | An applicant qualifies for the 2026 grant when all three conditions

 Embedding  hit@3 = ✓
  1. KZ-05    0.651  Astana | Astana is the capital city of Kazakhstan. With a population of 1,7 mil…
  2. KZ-06    0.507  Almaty (part 1) | Almaty is the most populous city in Kazakhstan, with more than two mil…
  3. KZ-03    0.419  Kazakhstan (part 3) | Kazakhstan has been inhabited since the Paleolithic era. In antiquity,…

Q-06: Where did the Soviet Union test its nuclear weapons?   relevant=['KZ-37', 'KZ-38']
 BM25  hit@3 = ✓
  1. KZ-37   19.090  Semipalatinsk Test Site (part 1) | The Semipalatinsk Test Site or Semipalatinsk-21 (Russian: Семипалатинс…
  2. KZ-38   17.168  Semipalatinsk Test Site (part 2) | The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 194…
  3. KZ-03    9.352  Kazakhstan (part 3) | Kazakhstan has been inhabited since the Paleolithic era. In antiquity,…


 Embedding  hit@3 = ✓
  1. KZ-37    0.596  Semipalatinsk Test Site (part 1) | The Semipalatinsk Test Site or Semipalatinsk-21 (Russian: Семипалатинс…
  2. KZ-38    0.510  Semipalatinsk Test Site (part 2) | The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 194…
  3. KZ-10    0.331  Baikonur Cosmodrome (part 1) | The Baikonur Cosmodrome is a spaceport operated by Russia within Kazak…

Q-07: Can the grant be sent to my mother's card?   relevant=['GO-15']
 BM25  hit@3 = ✗
  1. GO-21   14.652  Contacting the Grant Office | The Grant Office is in room 214 of the main building and is open Monda…
  2. GO-16    9.160  Combining the grant with other support | The need-based grant can be held together with a merit scholarship, bu…
  3. GO-01    9.053  About the need-based study grant 2026 | The need-based study grant 2026 is run by the Grant Office for full-ti…
 Embedding  hit@3 = ✓
  1. GO-15    0.422  Выплата гранта | Грант выплачивается двумя равными частями — в октябре и в фе

 Embedding  hit@3 = ✓
  1. GO-17    0.559  Academic progress during the year | The grant continues only while the student keeps up their studies. If …
  2. GO-18    0.459  Academic leave and transfers | A student who goes on academic leave keeps their grant decision, but p…
  3. GO-15    0.453  Выплата гранта | Грант выплачивается двумя равными частями — в октябре и в феврале — на…

Q-09: Who has to sign form GR-210?   relevant=['GO-07']
 BM25  hit@3 = ✓
  1. GO-07   10.243  Form GR-210: the income declaration | Form GR-210 is the household income declaration. It lists every adult …
  2. GO-03    7.991  Income bands | The income band is set from the monthly household income per person, d…
  3. GO-08    7.502  Form GR-305: appeals | Form GR-305 is used to appeal a decision of the Grant Office. It is th…


 Embedding  hit@3 = ✓
  1. GO-07    0.690  Form GR-210: the income declaration | Form GR-210 is the household income declaration. It lists every adult …
  2. GO-10    0.499  What counts as proof of income | Proof of income must accompany form GR-210. Accepted: a salary certifi…
  3. GO-13    0.468  Changes in circumstances | A student whose household income changes during the year must report i…

Q-10: Где находится грантовый отдел и в какие часы он работает?   relevant=['GO-21']
 BM25  hit@3 = ✗
  1. GO-15   11.603  Выплата гранта | Грант выплачивается двумя равными частями — в октябре и в феврале — на…
  2. GO-23   10.747  Апелляция: кратко | Если студент не согласен с решением, он может подать апелляцию по форм…
  3. KZ-01    0.000  Kazakhstan (part 1) | Kazakhstan, officially the Republic of Kazakhstan, is a landlocked cou…
 Embedding  hit@3 = ✓
  1. GO-21    0.477  Contacting the Grant Office | The Grant Office is in room 214 of the main building and is open Monda…
  2. GO-14    0

 Embedding  hit@3 = ✓
  1. KZ-14    0.720  Kazakh language (part 3) | In October 2017, the then president, Nursultan Nazarbayev, decreed tha…
  2. KZ-43    0.693  Kazakh alphabets | The Kazakh language has been written mainly in four scripts at various…
  3. KZ-05    0.434  Astana | Astana is the capital city of Kazakhstan. With a population of 1,7 mil…



In [7]:
print("| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |")
print("|---|---|:---:|---|:---:|")
totals = dict.fromkeys(RETRIEVERS, 0)
for q in all_queries:
    cells = []
    for name in RETRIEVERS:
        results = b_results[q["id"], name]
        ok = hit(results, q["relevant"])
        totals[name] += ok
        cells += [", ".join(c["id"] for c, _ in results), "✓" if ok else "✗"]
    print(f"| {q['id']} | " + " | ".join(cells) + " |")
n = len(all_queries)
print(f"| **hit@3 total** | | {totals['BM25']}/{n} | | {totals['Embedding']}/{n} |")
print(f"\nfixed eight only: BM25 {sum(hit(b_results[q['id'], 'BM25'], q['relevant']) for q in queries)}/8, "
      f"Embedding {sum(hit(b_results[q['id'], 'Embedding'], q['relevant']) for q in queries)}/8")

print("\nBM25 score per matched query term:")
for qid, chunk_ids in [("Q-01", ["GO-08", "GO-23"]), ("Q-02", ["KZ-17", "GO-01"]), ("Q-04", ["GO-05", "GO-02"]),
                       ("Q-07", ["GO-21", "GO-15"]), ("Q-10", ["GO-15", "GO-21"]), ("Q-11", ["KZ-43", "KZ-14"])]:
    text = next(q["text"] for q in all_queries if q["id"] == qid)
    for cid in chunk_ids:
        print(f"  {qid} → {cid}: {bm25_terms(text, cid)}")

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-02 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-07, GO-03, GO-08 | ✓ | GO-07, GO-10, GO-13 | ✓ |
| Q-10 | GO-15, GO-23, KZ-01 | ✗ | GO-21, GO-14, GO-15 | ✓ |
| Q-11 | KZ-43, KZ-14, KZ-06 | ✓ | KZ-14, KZ-43, KZ-05 | ✓ |
| **hit@3 total** | | 7/11 | | 11/11 |

fixed eight only: BM25 5/8, Embedding 8/8

BM25 score per matched query term:
  Q-01 → GO-08: {'is': 1.62, 'form': 3.3, 'gr': 2.58, '305': 4.84, 'used': 3.3, 'for': 0.19}
  Q-01 →

## B3 — comparison

The table below is copied from the output of the cell above.

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-02 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-07, GO-03, GO-08 | ✓ | GO-07, GO-10, GO-13 | ✓ |
| Q-10 | GO-15, GO-23, KZ-01 | ✗ | GO-21, GO-14, GO-15 | ✓ |
| Q-11 | KZ-43, KZ-14, KZ-06 | ✓ | KZ-14, KZ-43, KZ-05 | ✓ |
| **hit@3 total** | | 7/11 | | 11/11 |

On the eight fixed queries only: BM25 5/8, embeddings 8/8.

**Both retrievers agree:** which query, and why was it easy?

> **Q-06**, "Where did the Soviet Union test its nuclear weapons?". Both put KZ-37 and KZ-38 (Semipalatinsk Test Site) on ranks 1 and 2. BM25 scores are 19.09 and 17.17, and the next chunk has only 9.35. Embedding scores are 0.596 and 0.510, the next is 0.331. It was easy because the chunks use the same words as the query (`Soviet`, `nuclear`, `test`), these words are rare in the corpus, and the meaning matches too. When the words and the meaning point to the same chunk, both methods win.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> By hit@3 BM25 never beat embeddings in my runs: embeddings found a relevant chunk for all 11 queries. The best case for BM25 is **Q-11**, on rank 1. BM25 put KZ-43 (the relevant chunk, "Kazakh alphabets") first with 25.97, ahead of KZ-14 with 24.06. The difference comes from the exact string `cyrillic`: it gives 5.75 in KZ-43 but only 3.19 in KZ-14, because KZ-43 says "Cyrillic" three times. The embedding model put KZ-14 first (0.720 vs 0.693). Both chunks are about the same topic (the alphabet switch), so in vector space they look almost the same, and the model blurred them. Only KZ-43 says clearly that "the target year ... was pushed back to 2031".
>
> A smaller example is **Q-01**. The token `305` gives 4.84 to GO-08, so BM25 puts it far ahead (15.83 vs 8.20 for the next one). Exact codes like form numbers are where BM25 is strong. Embeddings also found GO-08 here, so it is not a win on hit@3.

**Embeddings did better:** which query, and what could BM25 not match?

> **Q-07**, "Can the grant be sent to my mother's card?". The answer is only in GO-15, which is in Russian: "перевод на счёт родителей или других лиц невозможен" (no transfer to the account of parents or other people). Embeddings put GO-15 first (0.422). BM25 shares **no token** with GO-15 (the term breakdown for GO-15 is `{}`), because of three gaps at once: another language, `mother` vs `родителей` (parents), and `card` vs `счёт` (account). Instead BM25 put GO-21 first (14.65), only because of `can`, `be`, `sent`, `to`, from the sentence "Questions can also be sent through the portal". That chunk is about contacting the office, not about payment.
>
> Q-10 is the same story (Russian query, English answer): BM25 matched only `и` and `в` in GO-15. Q-02 is a paraphrase case: the query says `poor` and `financial help`, the handbook says `low-income household`. BM25 put KZ-17 (Lake Balkhash) first on `there`, `like` and `is`.
>
> **Q-04** is interesting. BM25 put **GO-05** first, and its title is "Archived: the 2025 scheme (no longer in force)". It won on the word `minimum` (4.57 points), which GO-02 does not contain at all (GO-02 says "at least 2.67"). So the exact word the student typed points to the old rule with GPA 3.0. Embeddings put GO-02 first, but GO-05 is still second.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [8]:
import tiktoken
from openai import OpenAI

MODEL = "gpt-5.6-luna"
client = OpenAI()
enc = tiktoken.get_encoding("o200k_base")

SYSTEM_PROMPT = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""


def n_tokens(text):
    return len(enc.encode(text))


def format_context(chunks):
    return "\n\n".join(f"[{i}] {c['title']}\n{c['text']}" for i, c in enumerate(chunks, 1))


def rag_messages(question, chunks):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{format_context(chunks)}\n\nQuestion: {question}"},
    ]


def ask(messages):
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content, response.usage


def retrieve(question, search, k=3):
    return [chunk for chunk, _ in search(question, k)]


def print_messages(messages):
    for m in messages:
        print(f"===== {m['role']} =====\n{m['content']}\n")

## C1 — the complete prompt, printed before it is sent

In [9]:
c1 = qa_by_id["C-01"]
c1_chunks = retrieve(c1["question"], emb_search)
c1_messages = rag_messages(c1["question"], c1_chunks)
print_messages(c1_messages)

context_text = format_context(c1_chunks)
counts = {
    "System instructions": n_tokens(SYSTEM_PROMPT),
    "Question": n_tokens(c1["question"]),
    "Retrieved context": n_tokens(context_text),
    "Wrapper ('Context:' / 'Question:')": n_tokens(c1_messages[1]["content"]) - n_tokens(context_text) - n_tokens(c1["question"]),
}
counts["Total input (tiktoken)"] = sum(counts.values())

c1_answer, c1_usage = ask(c1_messages)
counts["usage.prompt_tokens"] = c1_usage.prompt_tokens

print("===== answer =====\n" + c1_answer + "\n")
for part, tokens in counts.items():
    print(f"{part:<38} {tokens:5d}")
print(f"\nchunks: {len(c1_chunks)}")
for chunk in c1_chunks:
    block = f"{chunk['title']}\n{chunk['text']}"
    print(f"  {chunk['id']} ({chunk['lang']}): {n_tokens(block)} tokens, {len(block)} chars")
print(f"context: {n_tokens(context_text)} tokens, {len(context_text)} chars")
print(f"gap: {counts['usage.prompt_tokens'] - counts['Total input (tiktoken)']} tokens")

===== system =====
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

===== user =====
Context:
[1] How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Income bands
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their student

===== answer =====
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

System instructions                       44
Question                                  18
Retrieved context                        302
Wrapper ('Context:' / 'Question:')         4
Total input (tiktoken)                   368
usage.prompt_tokens                      378

chunks: 3
  GO-04 (en): 96 tokens, 411 chars
  GO-03 (en): 107 tokens, 421 chars
  GO-15 (ru): 91 tokens, 349 chars
context: 302 tokens, 1197 chars
gap: 10 tokens


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 302 |
| Wrapper text (`Context:` / `Question:`) | 4 |
| **Total input** | **368** |
| **`usage.prompt_tokens` from the response** | **378** |

- Number of retrieved chunks: 3 (GO-04, GO-03, GO-15)
- Tokens per chunk (title + text): GO-04 96, GO-03 107, GO-15 91 (the Russian one)
- Total context size (tokens / characters): 302 tokens / 1197 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> The API counted 10 tokens more (378 vs 368). tiktoken only counts the text I wrote. The API does not send plain text to the model. It wraps every message in a chat format with special tokens: a start marker, the role name (`system`, `user`), and an end marker. I have two messages, so that is a few tokens each, plus a few more tokens that start the assistant's reply. Same as in HW1: the model pays for the format, not only for my words. Also, I assume `o200k_base` is the tokenizer of `gpt-5.6-luna`, which OpenAI does not confirm for every model.

**What exactly does retrieval add to the model's input?**

> Only text. The three chunks (title and text) are pasted into the user message as blocks `[1]`, `[2]`, `[3]`, between `Context:` and `Question:`. That is 302 tokens and 1197 characters, which is 82% of the 368 tokens I counted. The model gets no files, no links, no vectors and no scores. It does not even know which retriever chose the chunks or in which order they scored. The Russian chunk GO-15 goes in as Russian text, as it is.

## C2 — three systems, six questions

In [10]:
SYSTEMS = {
    "LLM only": lambda q: [{"role": "user", "content": q}],
    "BM25 RAG": lambda q: rag_messages(q, retrieve(q, bm25_search)),
    "Embedding RAG": lambda q: rag_messages(q, retrieve(q, emb_search)),
}

c2 = {}
for item in qa:
    print(f"##### {item['id']}: {item['question']}\nreference: {item['reference']}\n")
    for name, build in SYSTEMS.items():
        answer, _ = ask(build(item["question"]))
        c2[item["id"], name] = answer
        print(f"--- {name} ---\n{answer}\n")

##### C-01: How much does a student in income band 2 receive, and when is it paid?
reference: 150,000 tenge for the academic year, in two equal instalments: October and February.



--- LLM only ---
Which student funding scheme or country do you mean? “Income Band 2” is used by several programs, and the amount and payment schedule vary. Please share the scheme name or a link, plus the academic year.



--- BM25 RAG ---
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]



--- Embedding RAG ---
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

##### C-02: My grant application was refused. How long do I have to appeal, and which form do I use?
reference: 14 calendar days from the date on the decision letter, using form GR-305.



--- LLM only ---
The deadline and form depend on **which grant, funder, and country** are involved. The refusal letter or email should state:

- whether an appeal or reconsideration is allowed;
- the deadline—often counted from the date of the decision letter;
- the required form or submission method; and
- the grounds and supporting documents permitted.

If the letter does not specify these, contact the funder’s grants or appeals team immediately and ask for its **appeal/reconsideration procedure**. Do not assume that a general complaints form is sufficient, and keep proof of submission.

Please share the **grant scheme/funder, country, and date of the refusal letter** (you can remove personal details), and I can help identify the exact deadline and form.



...

--- BM25 RAG ---
Narxoz University was founded in 1963 as the Almaty Institute of National Economy. [1]



--- Embedding RAG ---
Narxoz University was founded in 1963 as the **Almaty Institute of National Economy**. [1]

##### C-06: I am a master's student. How much need-based grant can I get?
reference: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.



--- LLM only ---
The amount depends mainly on your **country, school, program, financial need, and FAFSA/financial-aid information**.

If you’re in the **United States**:

- **Federal Pell Grants generally are not available for master’s students**; they are primarily for undergraduates.
- You may still qualify for:
  - **University need-based grants or scholarships**
  - **State grants**—rules vary and many exclude graduate students
  - **TEACH Grant**, if you’re preparing to teach in a qualifying high-need field and agree to the service requirements
  - **Federal Work-Study**, if your school participates
  - **Assistantships**, which may provide a stipend and tuition remission
- Federal **Direct Unsubsidized Loans** and **Graduate PLUS Loans** are available, but these are loans—not grants.

A school usually calculates need roughly as:

**Cost of attendance − your Student Aid Index (SAI) − other aid = remaining demonstrated need**

There is no single standard grant amount; it could be 

--- BM25 RAG ---
I don’t know. The context says the need-based grant is only for full-time undergraduate students, while master’s students are covered by a separate scheme not described here. [3]



--- Embedding RAG ---
I do not know how much you can get. The need-based grant described is only for full-time undergraduate students; master’s students are covered by a separate scheme and should ask their department for its rules. [1]



Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ asks which scheme or country, gives no amount | ✓ 150,000 tenge, two equal instalments, October and February [1] | ✓ 150,000 tenge, two instalments of 75,000, October and February [1] |
| C-02 | ✗ "depends on the grant, funder and country", asks for details | ✓ 14 calendar days from the decision letter, form GR-305 [2][3] | ✓ 14 calendar days from the decision letter, form GR-305 [1][2] |
| C-03 | ✗ "which grant do you mean?", talks about the US Pell Grant and 2.0 | ✓ 2.67 on the 4.0 scale [2] | ✓ 2.67 on a 4.0 scale [1] |
| C-04 | ✓ decree October 2017, first target 2025, now 2031 (from memory) | ✓ October 2017, first 2025, now 2031 [1][2] | ✓ October 2017, first 2025, now 2031 [1][2] |
| C-05 | ✓ 1963, Alma-Ata Institute of National Economy (Alma-Ata is the old name of Almaty) | ✓ 1963, Almaty Institute of National Economy [1] | ✓ 1963, Almaty Institute of National Economy [1] |
| C-06 | ✗ long answer about US grants, Pell, TEACH, FAFSA | ✓ "I don't know", master's students have a separate scheme [3] | ✓ "I do not know", separate scheme, ask your department [1] |
| **agrees with `reference`** | 2/6 | 6/6 | 6/6 |

The LLM alone only got the two Wikipedia questions right (C-04, C-05). It has never seen the handbook, so for C-01, C-02, C-03 it did not invent numbers, it asked which scheme I mean. For C-06 it answered a different question (US grants). Neither RAG answer mentions that the C-04 target was moved "in January 2021", but the question only asks when the switch is now due, so I marked them ✓.

## C3 — question 1 of 3: C-03 (minimum GPA)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (15.80), GO-02 (14.06), GO-06 (13.85). The right chunk GO-02 is only second. GO-16 ("Combining the grant with other support") won because the words `need` and `based` give it 4.61 + 3.99, since it says "need-based grant" twice. GO-02 gets nothing for `need`/`based` and wins on `qualify` (4.02) and `gpa` (2.62).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-02 (0.629), GO-20 (0.569), GO-05 (0.545). The right chunk is first, but the archived 2025 rule GO-05 (minimum GPA 3.0) is in the context too.

**3. What was sent to the model** (the context blocks):

> BM25 RAG: [1] GO-16 (support cap, 400,000 tenge), [2] GO-02 ("cumulative GPA is at least 2.67 on the 4.0 scale"), [3] GO-06 (form GR-104).
> Embedding RAG: [1] GO-02 (2.67), [2] GO-20 (who cannot apply), [3] GO-05 ("ARCHIVED ... Under the 2025 scheme the minimum GPA for the grant was 3.0 ... do not apply to any application made for the 2026 academic year").
> The full blocks are printed in the cell below.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> More accurate and more specific. LLM only did not give a number for this grant, it asked which grant I mean and talked about the US Pell Grant and "around 2.0". Both RAG systems answered "2.67 on the 4.0 scale" and cited the block with GO-02 ([2] for BM25, [1] for embeddings).

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> This is the question where BM25 was worse than I expected. Its best chunk, GO-16, has nothing about GPA. With K=3 it did not matter, because GO-02 was second. But with K=1 (Final question 8) BM25 sent only GO-16 and the model answered "I do not know based on the provided context." That is a **retrieval failure**: the fact "at least 2.67" was not in the context, and the model correctly refused.
>
> Generation did well here. The embedding context had both 2.67 (GO-02) and the old 3.0 (GO-05), and the model chose 2.67. It read the word "ARCHIVED" and "do not apply to any application made for the 2026 academic year". Nothing was missed by generation.

In [11]:
def dissect(qid):
    item = qa_by_id[qid]
    print(f"{qid}: {item['question']}\nreference: {item['reference']}  relevant={item['relevant']}\n")
    for name, search in RETRIEVERS.items():
        results = search(item["question"])
        print(f"{name} top 3:")
        show(results)
        if name == "BM25":
            for chunk, _ in results:
                print(f"     {chunk['id']} terms: {bm25_terms(item['question'], chunk['id'])}")
    for name, search in [("BM25 RAG", bm25_search), ("Embedding RAG", emb_search)]:
        print(f"\n===== context sent by {name} =====\n{format_context(retrieve(item['question'], search))}")
    for name in SYSTEMS:
        print(f"\n--- {name} answer ---\n{c2[qid, name]}")


dissect("C-03")

C-03: What is the minimum GPA needed to qualify for the need-based grant?
reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)  relevant=['GO-02']

BM25 top 3:
  1. GO-16   15.799  Combining the grant with other support | The need-based grant can be held together with a merit scholarship, bu…
  2. GO-02   14.059  Who qualifies | An applicant qualifies for the 2026 grant when all three conditions ho…
  3. GO-06   13.852  Form GR-104: the application form | Form GR-104 is the application form for the need-based study grant. It…
     GO-16 terms: {'is': 1.42, 'the': 1.88, 'need': 4.61, 'based': 3.99, 'grant': 2.03}
     GO-02 terms: {'is': 1.65, 'the': 1.85, 'gpa': 2.62, 'qualify': 4.02, 'for': 0.17, 'grant': 1.89}
     GO-06 terms: {'is': 1.78, 'the': 1.96, 'to': 1.37, 'for': 0.28, 'need': 2.82, 'based': 2.44, 'grant': 1.24}
Embedding top 3:
  1. GO-02    0.629  Who qualifies | An applicant qualifies for the 2026 grant when all thre


===== context sent by Embedding RAG =====
[1] Who qualifies
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

[2] Who cannot apply
The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules.

[3] Archived: the 2025 scheme (no longer in force)
ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0

## C3 — question 2 of 3: C-06 (master's student)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (17.00), GO-01 (14.17), GO-20 (13.72). The only useful chunk, GO-20, is third. GO-16 and GO-01 are first because they repeat `need`, `based`, `grant` and `can`. GO-20 gets the most from `master` (4.29).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-20 (0.666), GO-16 (0.616), GO-01 (0.554). The same three chunks, but GO-20 is first.

**3. What was sent to the model** (the context blocks):

> BM25 RAG: [1] GO-16 (combining with other support, cap 400,000 tenge), [2] GO-01 (the grant is "for full-time undergraduate students"), [3] GO-20 ("Master's and doctoral students are supported by a separate scheme, which this handbook does not describe").
> Embedding RAG: the same chunks in the order GO-20, GO-16, GO-01.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> More grounded. There is no answer in the corpus, and the correct response is "I don't know". LLM only wrote a long answer about US Pell Grants, TEACH Grants and FAFSA, which has nothing to do with this university. Both RAG systems said they do not know the amount and explained why: master's students have a separate scheme (BM25 cited [3], embeddings cited [1], both GO-20). The context also had a tempting number, "400,000 tenge" in GO-16, but neither RAG answer used it as an amount.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed. Retrieval brought the only relevant chunk, GO-20, in both systems. Generation followed the system message "If the answer cannot be found in the context, say that you do not know". For example, BM25 RAG: "I don't know. The context says the need-based grant is only for full-time undergraduate students, while master's students are covered by a separate scheme not described here." If GO-20 had dropped out of the top 3 (with BM25 it was only third), the model would still have no amount, but it would lose the reason, so this would be a retrieval problem, not a generation one.

In [12]:
dissect("C-06")

C-06: I am a master's student. How much need-based grant can I get?
reference: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.  relevant=['GO-20']

BM25 top 3:
  1. GO-16   16.996  Combining the grant with other support | The need-based grant can be held together with a merit scholarship, bu…
  2. GO-01   14.173  About the need-based study grant 2026 | The need-based study grant 2026 is run by the Grant Office for full-ti…
  3. GO-20   13.720  Who cannot apply | The need-based grant is for full-time undergraduate students only. Par…
     GO-16 terms: {'a': 1.62, 'student': 1.78, 'need': 4.61, 'based': 3.99, 'grant': 2.03, 'can': 2.98}
     GO-01 terms: {'a': 1.67, 'student': 2.62, 'need': 2.72, 'based': 2.35, 'grant': 2.09, 'can': 2.72}
     GO-20 terms: {'a': 1.14, 'master': 4.29, 's': 0.84, 'need': 3.23, 'based': 2.8, 'grant': 1.42}
Embedding top 3:
  1. GO-20    0.666  Who cannot apply | The need-based grant is for fu

## C3 — question 3 of 3: C-04 (Kazakh alphabet)

**1. What BM25 retrieved** (top 3, ids and scores):

> KZ-43 (25.97), KZ-14 (24.06), KZ-06 (16.84). KZ-43 is the relevant chunk. KZ-06 (Almaty) is noise: it got points from `when`, `now`, `kazakhstan`, which are common words here.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> KZ-14 (0.720), KZ-43 (0.693), KZ-05 (0.434). The relevant chunk KZ-43 is only second. KZ-05 (Astana) is noise.

**3. What was sent to the model** (the context blocks):

> BM25 RAG: [1] KZ-43 ("In October 2017, a presidential decree ... by 2025. In January 2021, the target year for finishing the transition was pushed back to 2031."), [2] KZ-14 ("In October 2017 ... by 2025 ... It is scheduled to be phased in from 2023 to 2031."), [3] KZ-06 (Almaty).
> Embedding RAG: [1] KZ-14, [2] KZ-43, [3] KZ-05 (Astana).

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> More grounded, but not more accurate. This is a Wikipedia question, and the model already knew the answer: LLM only said "October 2017", first target 2025, now 2031. Both RAG answers say the same thing, with citations [1][2]. Here retrieval did not add a new fact, it added a source that I can check. This is the "fair test" from the README: for text the model has read in training, RAG mostly adds proof, not knowledge.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Both RAG answers missed "January 2021", the date when the target was moved. This fact **was in the context** in both systems: KZ-43 says "In January 2021, the target year for finishing the transition was pushed back to 2031", and it was block [1] for BM25 and [2] for embeddings. So this is a small **generation** miss, not retrieval: the model wrote "was later postponed" instead of the date. It is small because the question did not ask when the target was moved. The noise chunks (KZ-06, KZ-05) did not hurt, the model ignored them.

In [13]:
dissect("C-04")

C-04: When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
reference: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.  relevant=['KZ-43']

BM25 top 3:
  1. KZ-43   25.966  Kazakh alphabets | The Kazakh language has been written mainly in four scripts at various…
  2. KZ-14   24.062  Kazakh language (part 3) | In October 2017, the then president, Nursultan Nazarbayev, decreed tha…
  3. KZ-06   16.835  Almaty (part 1) | Almaty is the most populous city in Kazakhstan, with more than two mil…
     KZ-43 terms: {'kazakhstan': 0.48, 'to': 1.52, 'the': 1.81, 'kazakh': 0.82, 'alphabet': 3.52, 'from': 0.34, 'cyrillic': 5.75, 'latin': 4.05, 'and': 1.52, 'is': 1.31}
     KZ-14 terms: {'to': 1.58, 'the': 1.85, 'kazakh': 0.74, 'alphabet': 4.63, 'from': 0.44, 'cyrillic': 3.19, 'latin': 3.78, 'and': 1.58, 'is': 1.23}
     KZ-06 terms: {'when': 1.64, 'kazakhstan': 0.45

Embedding top 3:
  1. KZ-14    0.720  Kazakh language (part 3) | In October 2017, the then president, Nursultan Nazarbayev, decreed tha…
  2. KZ-43    0.693  Kazakh alphabets | The Kazakh language has been written mainly in four scripts at various…
  3. KZ-05    0.434  Astana | Astana is the capital city of Kazakhstan. With a population of 1,7 mil…

===== context sent by BM25 RAG =====
[1] Kazakh alphabets
The Kazakh language has been written mainly in four scripts at various points of time—Old Turkic, Cyrillic, Latin, and Arabic—each having a distinct alphabet. The Arabic script is used in Iran, Afghanistan, and China, while the Cyrillic script is used in Kazakhstan, Kyrgyzstan, Russia, and Mongolia. In October 2017, a presidential decree in Kazakhstan ordered a transition from the Cyrillic to Latin script to be implemented by 2025. In January 2021, the target year for finishing the transition was pushed back to 2031.

[2] Kazakh language (part 3)
In October 2017, the then president, 


===== context sent by Embedding RAG =====
[1] Kazakh language (part 3)
In October 2017, the then president, Nursultan Nazarbayev, decreed that the writing system would change from Cyrillic to Latin script by 2025. The proposed Latin alphabet has been revised several times and, as of January 2021, is close to the inventory of the Turkish alphabet, though it lacks the letters C and Ç and has four additional letters: Ä, Ñ, Q, and Ū (though other letters, such as Y, have different values in the two languages). It is scheduled to be phased in from 2023 to 2031. Over one million Kazakh speakers in Xinjiang use a modified version of the Perso-Arabic script for writing.

[2] Kazakh alphabets
The Kazakh language has been written mainly in four scripts at various points of time—Old Turkic, Cyrillic, Latin, and Arabic—each having a distinct alphabet. The Arabic script is used in Iran, Afghanistan, and China, while the Cyrillic script is used in Kazakhstan, Kyrgyzstan, Russia, and Mongolia. In Oc

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 counts the exact words that the query and the chunk share, and gives more weight to rare words. Embedding retrieval turns the query and the chunk into vectors and compares their meaning, so it can match chunks with no shared words. In Part A, BM25 gave 0 to every document because `RAG` and `retrieve` are not written the same way in D2, while cosine put D2 first.

**2. When can BM25 perform better than embeddings?**

> When the query has an exact rare string: a form code, a number, a name. In my runs, `305` gave GO-08 a big lead in Q-01 (15.83 vs 8.20), and `cyrillic` made BM25 put KZ-43 above KZ-14 in Q-11, while embeddings mixed up those two similar chunks. BM25 is also free, fast and easy to explain: I can print which word gave how many points.

**3. When can embeddings perform better than BM25?**

> When the query uses other words or another language than the chunk. In Q-07 the answer is in Russian (GO-15) and the query is in English, BM25 had zero shared tokens and missed. In Q-02 `poor` vs `low-income` is a paraphrase, BM25 returned a chunk about Lake Balkhash. In Q-10 (Russian query, English answer) BM25 only matched `и` and `в`. Embeddings found the right chunk in all three.

**4. Why is cosine similarity used with embeddings?**

> Cosine looks at the angle between two vectors, not their length. The direction of an embedding carries the meaning, while the length can change with things like text length. In Part A, D2 has a much bigger dot product than D1 (1.08 vs 0.78), but after dividing by the lengths they are almost equal (0.9977 vs 0.9956). EmbeddingGemma also returns normalized vectors, so cosine is the same as a simple dot product and is cheap to compute.

**5. What information is actually sent to an LLM in a RAG system?**

> Only text: the system message, the retrieved chunks pasted as numbered blocks, and the question. In C1 it was 44 + 302 + 18 tokens (plus 4 tokens of my labels), and the API counted 378 with the chat format. Nothing else is sent: no documents, no vectors, no scores.

**6. Does the LLM search the original documents itself?**

> No. My code does the search, picks the top chunks and pastes them into the prompt. The model only continues the text it gets. It has never seen the other 71 chunks. In the K=1 test BM25 sent only GO-16, and the model said "I do not know", even though the answer (GO-02) was in the corpus.

**7. What happens if retrieval returns an irrelevant chunk?**

> It takes space in the prompt and can push the model to a wrong answer. If the irrelevant chunk is the only one, the model cannot answer at all (C-03 with BM25 and K=1). If it sits next to the right chunk, a good model usually ignores it: in C-03 the archived GO-05 with GPA 3.0 was in the embedding context, and the model still answered 2.67. A chunk with a wrong but believable fact, like GO-05, is the most dangerous kind.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> On C-03 (minimum GPA), output in the cell below:
>
> | K | BM25 chunks | BM25 answer | prompt tokens | Embedding chunks | Embedding answer | prompt tokens |
> |---|---|---|---:|---|---|---:|
> | 1 | GO-16 | "I do not know" ✗ | 153 | GO-02 | 2.67 ✓ | 180 |
> | 3 | GO-16, GO-02, GO-06 | 2.67 ✓ | 356 | GO-02, GO-20, GO-05 | 2.67 ✓ | 346 |
> | 8 | + GO-05 (archived 3.0) and 4 more | 2.67 ✓ | 860 | + GO-16, GO-01 and 3 more | 2.67 ✓ | 816 |
>
> A bigger K helps when the right chunk is not first: BM25 went from "I do not know" at K=1 to correct at K=3. But a bigger K also brings more noise and costs more. At K=8 the prompt is about 2.4 times longer than at K=3 and adds the archived GO-05 with the old GPA 3.0, with no gain in the answer. This model was not confused this time, but each extra wrong chunk is one more chance for a wrong answer.

In [14]:
topk_question = qa_by_id["C-03"]["question"]
for name, search in [("BM25", bm25_search), ("Embedding", emb_search)]:
    for k in (1, 3, 8):
        chunks = retrieve(topk_question, search, k)
        messages = rag_messages(topk_question, chunks)
        answer, usage = ask(messages)
        print(f"--- {name}, K={k}: {[c['id'] for c in chunks]}, prompt_tokens={usage.prompt_tokens}\n{answer}\n")

--- BM25, K=1: ['GO-16'], prompt_tokens=153
I do not know based on the provided context.



--- BM25, K=3: ['GO-16', 'GO-02', 'GO-06'], prompt_tokens=356
The minimum GPA is **2.67 on a 4.0 scale**. [2]



--- BM25, K=8: ['GO-16', 'GO-02', 'GO-06', 'GO-05', 'GO-01', 'GO-11', 'GO-03', 'GO-17'], prompt_tokens=860
The minimum GPA needed is **2.67 on a 4.0 scale**. [2]



--- Embedding, K=1: ['GO-02'], prompt_tokens=180
The minimum GPA needed is **2.67 on a 4.0 scale**. [1]



--- Embedding, K=3: ['GO-02', 'GO-20', 'GO-05'], prompt_tokens=346
The minimum GPA is **2.67 on a 4.0 scale**. [1]



--- Embedding, K=8: ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17'], prompt_tokens=816
The minimum GPA needed to qualify is **2.67 on a 4.0 scale**. [1]



**9. Is a vector database required to build a RAG system?**

> No. My notebook keeps all 74 embeddings in one NumPy array in memory (`doc_embeddings`, shape 74 × 768, float32) and compares the query with every chunk using `embedder.similarity`. The BM25 index is just a Python object from `rank_bm25`. For 74 chunks this takes milliseconds. A vector database becomes useful when there are millions of chunks, or when you need saving to disk, updates and filters.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> Embeddings worked better for retrieval. hit@3 over my 11 queries: **embeddings 11/11, BM25 7/11**. On the eight fixed queries: embeddings 8/8, BM25 5/8. BM25 missed Q-02 (paraphrase), Q-04 (it put the archived GO-05 first), Q-07 and Q-10 (Russian text). In C2 both RAG systems agreed with the reference on **6/6**, and LLM only on 2/6. BM25 still got 6/6 because for every C question the right chunk was somewhere in its top 3, even if not first (GO-02 second for C-03, GO-20 third for C-06). So with K=3 the model fixed BM25's bad ranking, but with K=1 BM25 failed on C-03 and embeddings did not.

---
# Conclusion

> Next I would build a hybrid retriever. I would run BM25 and EmbeddingGemma together and join the two rankings, for example with reciprocal rank fusion. My runs show they fail in different places: BM25 is strong on exact codes like `GR-305` and rare words like `Cyrillic`, embeddings are strong on paraphrases and on Russian and Kazakh text. I would also use the metadata the corpus already has and drop chunks marked as archived, like GO-05, before they reach the model, because that chunk uses the exact words students type and has the wrong rule. I would keep K around 3, because K=8 more than doubled the prompt size and added noise without better answers. Finally, I would add a check that every answer cites a block, so an answer with no source is easy to catch.